In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

orders_df = pd.DataFrame({
    'shop': ['Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town', 'Hilltop', 'Harbour', 'Riverside', 'Old Town'],
    'weekday': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
    'croissants': [99, 79, 89, 80, 73, 112, 112, 81, 66, 91, 67, 75, 126, 118, 95, 71, 85, 90, 97, 111, 123, 79, 98, 87, 92, 78, 98, 135, 81, 86, 85, 86, 93, 122, 116, 56, 89, 68, 77, 71, 117, 121],
    'loaves': [42, 53, 45, 43, 48, 60, 39, 44, 55, 32, 53, 54, 66, 57, 40, 58, 53, 52, 53, 60, 50, 44, 28, 57, 54, 50, 75, 63, 48, 51, 38, 49, 56, 52, 68, 50, 56, 49, 52, 35, 52, 52],
    'cakes': [22, 7, 13, 21, 15, 32, 35, 33, 12, 14, 11, 20, 31, 32, 35, 19, 21, 19, 22, 28, 33, 27, 24, 10, 12, 19, 26, 30, 26, 19, 22, 30, 22, 32, 37, 15, 26, 15, 10, 18, 25, 37],
    'temperature_c': [11.4, 4.9, 6.8, 23.7, 8.2, 19.6, 22.3, 26.5, 4.9, 17.3, 5.4, 4.5, 7.0, 22.8, 26.0, 4.1, 15.9, 13.0, 8.8, 4.6, 12.4, 11.9, 16.1, 10.0, 11.6, 9.2, 10.0, 9.1, 23.3, 21.3, 26.3, 24.8, 12.1, 14.0, 9.5, 17.4, 16.0, 20.7, 6.4, 25.0, 14.4, 16.9],
    'rain_mm': [4.0, 2.8, 2.2, 1.4, 0.0, 2.2, 0.8, 0.0, 0.0, 0.0, 0.2, 4.4, 1.5, 0.0, 2.6, 0.3, 0.0, 0.0, 2.9, 3.2, 5.7, 2.2, 3.8, 0.0, 2.4, 0.0, 0.0, 0.7, 0.0, 0.0, 0.0, 2.7, 0.0, 0.3, 0.0, 4.1, 4.7, 4.5, 0.0, 0.0, 0.0, 0.7],
    'promotion': [True, False, False, False, False, False, False, True, False, False, False, False, False, False, True, False, True, False, True, False, False, False, True, False, False, False, False, True, False, False, False, False, True, False, False, False, True, False, False, False, False, False],
    'complaints': [2, 2, 1, 3, 1, 1, 1, 2, 0, 1, 0, 1, 0, 2, 0, 0, 2, 1, 2, 2, 1, 1, 2, 0, 1, 1, 2, 0, 2, 2, 1, 1, 0, 2, 0, 1, 1, 0, 0, 1, 1, 2],
})

orders_df["total_items"] = orders_df[["croissants", "loaves", "cakes"]].sum(axis=1)
orders_df["is_weekend"] = orders_df["weekday"].isin(["Sat", "Sun"])
orders_df["had_complaint"] = orders_df["complaints"] > 0

In [ ]:
posterior_rng = np.random.default_rng(5)
posterior_draws = 4000


def beta_binomial_posterior(successes, trials, prior_alpha=1.0, prior_beta=1.0):
    alpha_post = prior_alpha + successes
    beta_post = prior_beta + trials - successes
    draws = posterior_rng.beta(alpha_post, beta_post, posterior_draws)
    lower, median, upper = np.quantile(draws, [0.025, 0.5, 0.975])
    return {
        "successes": int(successes),
        "trials": int(trials),
        "observed_rate": successes / trials if trials else float("nan"),
        "alpha_post": float(alpha_post),
        "beta_post": float(beta_post),
        "mean": float(alpha_post / (alpha_post + beta_post)),
        "median": float(median),
        "lower": float(lower),
        "upper": float(upper),
        "draws": draws,
    }


def prior_from_past_season(past_rate, strength):
    prior_alpha = max(past_rate * strength, 0.5)
    prior_beta = max((1 - past_rate) * strength, 0.5)
    return prior_alpha, prior_beta


overall_days = len(orders_df)
overall_complaint_days = int(orders_df["had_complaint"].sum())
flat_posterior = beta_binomial_posterior(overall_complaint_days, overall_days)
printable = {key: value for key, value in flat_posterior.items() if key != "draws"}
print(printable)
past_alpha, past_beta = prior_from_past_season(0.35, 20)
informed_posterior = beta_binomial_posterior(overall_complaint_days, overall_days, past_alpha, past_beta)
print({key: round(value, 4) if isinstance(value, float) else value for key, value in informed_posterior.items() if key != "draws"})
shop_rows_list = []
for shop_name, shop_rows in orders_df.groupby("shop"):
    shop_posterior = beta_binomial_posterior(int(shop_rows["had_complaint"].sum()), len(shop_rows), past_alpha, past_beta)
    shop_rows_list.append({"shop": shop_name, "days": len(shop_rows), "complaint_days": shop_posterior["successes"], "mean": round(shop_posterior["mean"], 4), "lower": round(shop_posterior["lower"], 4), "upper": round(shop_posterior["upper"], 4)})
    print(f"{shop_name:10s} complaint-day rate {shop_posterior['mean']:.3f} [{shop_posterior['lower']:.3f}, {shop_posterior['upper']:.3f}]")
shop_posterior_df = pd.DataFrame(shop_rows_list)
weekend_posterior = beta_binomial_posterior(int(orders_df.loc[orders_df["is_weekend"], "had_complaint"].sum()), int(orders_df["is_weekend"].sum()), past_alpha, past_beta)
weekday_posterior = beta_binomial_posterior(int(orders_df.loc[~orders_df["is_weekend"], "had_complaint"].sum()), int((~orders_df["is_weekend"]).sum()), past_alpha, past_beta)
weekend_minus_weekday = weekend_posterior["draws"] - weekday_posterior["draws"]
probability_weekend_worse = float((weekend_minus_weekday > 0).mean())
print(f"P(weekend complaint rate > weekday): {probability_weekend_worse:.3f}")
promotion_posterior = beta_binomial_posterior(int(orders_df.loc[orders_df["promotion"], "had_complaint"].sum()), int(orders_df["promotion"].sum()), past_alpha, past_beta)
regular_posterior = beta_binomial_posterior(int(orders_df.loc[~orders_df["promotion"], "had_complaint"].sum()), int((~orders_df["promotion"]).sum()), past_alpha, past_beta)
promotion_difference = promotion_posterior["draws"] - regular_posterior["draws"]
print(f"P(promotion days have more complaints): {(promotion_difference > 0).mean():.3f}")
sell_out_threshold = 120
orders_df["sold_out_croissants"] = orders_df["croissants"] >= sell_out_threshold
sell_out_rows = []
for weekday_name in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]:
    weekday_rows = orders_df[orders_df["weekday"] == weekday_name]
    weekday_posterior_result = beta_binomial_posterior(int(weekday_rows["sold_out_croissants"].sum()), len(weekday_rows))
    sell_out_rows.append((weekday_name, len(weekday_rows), weekday_posterior_result["mean"], weekday_posterior_result["lower"], weekday_posterior_result["upper"]))
for weekday_name, day_count, mean_rate, lower, upper in sell_out_rows:
    print(f"{weekday_name}: {day_count} days, sell-out chance {mean_rate:.3f} [{lower:.3f}, {upper:.3f}]")
future_days = 30
future_complaint_rates = informed_posterior["draws"]
future_complaint_days = posterior_rng.binomial(future_days, future_complaint_rates)
future_summary = {
    "expected": round(float(future_complaint_days.mean()), 2),
    "p10": int(np.quantile(future_complaint_days, 0.1)),
    "p90": int(np.quantile(future_complaint_days, 0.9)),
    "prob_more_than_15": round(float((future_complaint_days > 15).mean()), 4),
}
print("Next 30 days, complaint days:", future_summary)
risk_labels = []
for row in shop_posterior_df.itertuples(index=False):
    if row.lower > 0.5:
        risk_labels.append((row.shop, "high"))
    elif row.upper < 0.3:
        risk_labels.append((row.shop, "low"))
    else:
        risk_labels.append((row.shop, "uncertain"))
for shop_name, risk_label in risk_labels:
    print(f"{shop_name}: {risk_label} complaint risk")

In [ ]:
metric_rows = []
for shop_name, shop_rows in orders_df.groupby("shop"):
    weekend_share = shop_rows["is_weekend"].mean()
    mean_items = shop_rows["total_items"].mean()
    item_spread = shop_rows["total_items"].std()
    metric_rows.append({"shop": shop_name, "weekend_share": round(weekend_share, 3), "mean_items": round(mean_items, 1), "item_spread": round(item_spread, 1)})
metric_df = pd.DataFrame(metric_rows).merge(shop_posterior_df, on="shop")
print(metric_df)

In [ ]:
shrinkage_rows = []
for row in metric_df.itertuples(index=False):
    raw_rate = row.complaint_days / row.days
    shrinkage_rows.append((row.shop, round(raw_rate, 3), row.mean, round(raw_rate - row.mean, 3)))
shrinkage_df = pd.DataFrame(shrinkage_rows, columns=["shop", "raw", "posterior", "shrink"])
print(shrinkage_df)

In [ ]:
def plot_rate_forest(summary_df, title_text):
    figure, axis = plt.subplots(figsize=(6, 3))
    positions = np.arange(len(summary_df))
    axis.hlines(positions, summary_df["lower"], summary_df["upper"])
    axis.plot(summary_df["mean"], positions, "o")
    axis.set_yticks(positions)
    axis.set_yticklabels(summary_df["shop"])
    axis.set_xlabel("Complaint-day rate")
    axis.set_title(title_text)
    plt.tight_layout()
    return figure


forest_figure = plot_rate_forest(shop_posterior_df, "Complaint-day rate by shop")
plt.show()

In [ ]:
ordered_shops = [row.shop for row in shop_posterior_df.sort_values("mean").itertuples(index=False)]
print("Shops from lowest to highest complaint rate:", ordered_shops)

In [ ]:
headline = f"Across {overall_days} days the complaint-day rate is about {informed_posterior['mean']:.1%} (95% interval {informed_posterior['lower']:.1%} to {informed_posterior['upper']:.1%})."
print(headline)

In [ ]:
for weekday_name, day_count, mean_rate, lower, upper in sell_out_rows:
    if mean_rate > 0.3:
        print(f"Bake extra croissants on {weekday_name}: sell-out chance {mean_rate:.0%}")

In [ ]:
for shop_name, risk_label in risk_labels:
    if risk_label != "low":
        print(f"Review service at {shop_name} ({risk_label} risk)")

In [ ]:
print("Posterior draws per estimate:", posterior_draws, "| shops analysed:", len(shop_posterior_df))